**Link repo của tôi: https://github.com/ducphat1509-beep/artificial-intelligence**

# Báo cáo Bài tập tuần - Buổi 10: Các biến thể Hill Climbing và Local Beam Search cho 8-Puzzle
**Nguyễn Đức Phát - 24110296**

---

## 1. Mục tiêu và phạm vi của buổi học
Trong buổi học này, tôi tiếp tục nghiên cứu nhóm **Local Search (Tìm kiếm cục bộ)** trên bài toán mẫu **8-Puzzle**, nhằm giữ sự thống nhất với các buổi trước. Nếu các thuật toán như BFS, UCS, A*, Greedy hay IDA* thường tập trung vào việc tìm một đường đi từ trạng thái đầu đến trạng thái đích, thì Local Search nhìn bài toán theo hướng tối ưu hóa trạng thái hiện tại.

Với 8-Puzzle, mỗi trạng thái là một ma trận 3x3 gồm các ô `1..8` và ô trống `0`. Trạng thái đích được sử dụng trong bài:

```text
1 2 3
8 0 4
7 6 5
```

Trong các thuật toán Local Search, tôi không ưu tiên lưu toàn bộ đường đi tối ưu như A* hay IDA*. Thay vào đó, tôi đánh giá chất lượng trạng thái bằng hàm heuristic `h(n)` và cố gắng đưa `h(n)` về 0. Khi `h(n) = 0`, trạng thái hiện tại trùng với goal.

---

## 2. Hàm heuristic và ảnh hưởng đến hiệu năng
Tôi sử dụng hai cách tính heuristic để so sánh:

1. **Misplaced Tiles**: đếm số ô đang sai vị trí so với goal, không tính ô trống.
2. **Manhattan Distance**: tính tổng khoảng cách hàng/cột của từng ô đến vị trí đúng của nó.

Về mặt chất lượng đánh giá, **Manhattan Distance thường tốt hơn Misplaced Tiles** vì nó chứa nhiều thông tin hơn. Một ô sai vị trí có thể chỉ lệch 1 bước, nhưng cũng có thể lệch 3 hoặc 4 bước; Misplaced Tiles xem các trường hợp này gần như giống nhau, còn Manhattan phân biệt được mức độ sai lệch.

Ảnh hưởng thực nghiệm có thể quan sát trên giao diện:
- Với **Misplaced Tiles**, nhiều neighbor có cùng giá trị `h(n)`, dễ tạo plateau. Khi đó Stochastic HC hoặc Sideways HC có thể trông khá ngẫu nhiên vì có nhiều lựa chọn ngang giá.
- Với **Manhattan Distance**, giá trị heuristic mịn hơn, thường giúp thuật toán nhận ra hướng cải thiện rõ hơn, giảm số lần kẹt giả do đánh giá quá thô.
- Tuy nhiên, cả hai heuristic đều không biến Hill Climbing thành thuật toán đầy đủ hay tối ưu. Chúng chỉ thay đổi chất lượng “địa hình” mà thuật toán đang leo.

---

## 3. Steepest-Ascent Hill Climbing
### a. Ý tưởng
**Steepest-Ascent Hill Climbing** xét toàn bộ các neighbor của trạng thái hiện tại, sau đó chọn neighbor tốt nhất, tức neighbor có `h(n)` nhỏ nhất và phải tốt hơn trạng thái hiện tại.

Với 8-Puzzle, mỗi bước di chuyển ô trống tạo ra tối đa 4 neighbor. Thuật toán chọn bước có độ giảm heuristic mạnh nhất:

```text
chọn argmin h(neighbor), với điều kiện h(neighbor) < h(current)
```

Nếu không tồn tại neighbor tốt hơn, thuật toán dừng vì bị kẹt tại local optimum hoặc plateau.

### b. Bốn đặc trưng
- **Tính đầy đủ (Completeness)**: Không. Thuật toán có thể dừng trước khi tới goal nếu mọi neighbor đều không tốt hơn.
- **Tính tối ưu (Optimality)**: Không. Thuật toán không đảm bảo tìm đường đi ngắn nhất, thậm chí có thể không tìm được goal.
- **Độ phức tạp thời gian**: Xấp xỉ `O(t * b * cost(h))`, với `t` là số bước leo, `b` là số neighbor tối đa, và `cost(h)` là chi phí tính heuristic.
- **Độ phức tạp không gian**: `O(b)` nếu lưu danh sách neighbor; có thể gần `O(1)` nếu duyệt và giữ neighbor tốt nhất.

### c. Nhận xét hiệu năng
Steepest-Ascent thường ổn định hơn Simple Hill Climbing vì nó không chọn neighbor tốt hơn đầu tiên một cách vội vàng. Tuy nhiên, chính sự tham lam này cũng khiến nó dễ bị kẹt nếu hướng giảm `h(n)` trước mắt không dẫn tới goal.

---

## 4. Stochastic Hill Climbing
### a. Ý tưởng
**Stochastic Hill Climbing** không nhất thiết chọn neighbor tốt nhất. Thuật toán tạo tập các neighbor tốt hơn trạng thái hiện tại, rồi chọn ngẫu nhiên một trạng thái trong tập đó:

```text
Better = { neighbor | h(neighbor) < h(current) }
```

Nếu `Better` rỗng, thuật toán dừng.

### b. Bốn đặc trưng
- **Tính đầy đủ**: Không. Nếu bị kẹt tại plateau hoặc local optimum, thuật toán không có cơ chế quay lui.
- **Tính tối ưu**: Không. Việc chọn ngẫu nhiên giúp đa dạng đường chạy nhưng không đảm bảo đường đi ngắn nhất.
- **Độ phức tạp thời gian**: Xấp xỉ `O(t * b * cost(h))`.
- **Độ phức tạp không gian**: `O(b)` do cần lưu tập neighbor tốt hơn để chọn ngẫu nhiên.

### c. Nhận xét hiệu năng
Stochastic HC có thể tránh một số lựa chọn quá tham lam của Steepest-Ascent, nhưng đổi lại kết quả giữa các lần chạy có thể khác nhau. Với Misplaced Tiles, số neighbor cùng mức đánh giá thường nhiều hơn, nên tính ngẫu nhiên thể hiện rõ hơn. Với Manhattan, tập neighbor tốt hơn thường có phân hóa tốt hơn.

---

## 5. Hill Climbing with Sideways Moves
### a. Ý tưởng
Thay vì chỉ chọn neighbor tốt hơn, thuật toán cho phép chọn ngẫu nhiên trong tập neighbor tốt hơn hoặc bằng:

```text
Candidate = { neighbor | h(neighbor) <= h(current) }
```

Trong đó:
- `h(neighbor) < h(current)` là bước cải thiện thật sự.
- `h(neighbor) == h(current)` là bước đi ngang (*sideways move*) trên plateau.

Do bước đi ngang có thể gây lặp vô hạn, tôi đặt thêm `sideways_limit` để giới hạn số bước ngang liên tiếp.

### b. Bốn đặc trưng
- **Tính đầy đủ**: Không trong trường hợp giới hạn `sideways_limit`. Nếu không giới hạn, thuật toán còn có nguy cơ lặp vô hạn.
- **Tính tối ưu**: Không, vì thuật toán vẫn chỉ ra quyết định dựa trên thông tin cục bộ.
- **Độ phức tạp thời gian**: Xấp xỉ `O(t * b * cost(h))`, trong đó `t` có thể tăng do các bước đi ngang.
- **Độ phức tạp không gian**: `O(b)`.

### c. Nhận xét hiệu năng
Sideways Moves đặc biệt hữu ích khi heuristic tạo plateau. Với Misplaced Tiles, plateau xuất hiện nhiều hơn, nên biến thể này thường thể hiện rõ vai trò hơn. Với Manhattan, plateau ít hơn nhưng vẫn có thể xảy ra.

---

## 6. Random-Restart Hill Climbing (`max_restart`)
### a. Ý tưởng
**Random-Restart Hill Climbing** không cố sửa một lần chạy bị kẹt. Thay vào đó, thuật toán chấp nhận rằng một điểm bắt đầu có thể xấu, rồi thử lại từ nhiều trạng thái khởi tạo khác nhau.

Quy trình tôi cài đặt:
1. Lần chạy đầu tiên luôn xuất phát từ **initial state cố định** của bài toán, để bảo toàn đúng dữ kiện ban đầu.
2. Chạy Steepest-Ascent Hill Climbing từ initial đó.
3. Nếu bị kẹt, thuật toán mới xem như “quay xuống chân núi” và sinh một start state hợp lệ khác bằng cách scramble từ goal.
4. Tiếp tục leo từ start mới, tối đa `max_restart` lần.
5. Dừng khi tìm thấy goal hoặc dùng hết số lần restart cho phép.

Điểm quan trọng là Random-Restart không phủ nhận initial state. Nó chỉ bổ sung quyền thử lại ở các vùng khác của không gian trạng thái sau khi lần leo hiện tại thất bại.

### b. Bốn đặc trưng
- **Tính đầy đủ**: Không nếu `max_restart` hữu hạn. Nếu cho restart vô hạn và xác suất sinh được vùng dẫn tới goal lớn hơn 0, xác suất thành công sẽ tăng dần.
- **Tính tối ưu**: Không. Thuật toán chỉ cố tìm goal, không chứng minh đường đi ngắn nhất.
- **Độ phức tạp thời gian**: Xấp xỉ `O((max_restart + 1) * t * b * cost(h))`.
- **Độ phức tạp không gian**: `O(b)` cho mỗi lần leo, cộng thêm phần log mô phỏng nếu giao diện lưu lịch sử.

### c. Nhận xét hiệu năng
Random-Restart thường cải thiện tỷ lệ tìm thấy goal rõ rệt so với một lần Hill Climbing đơn lẻ. Manhattan Distance thường giúp mỗi lần restart đi có định hướng hơn; Misplaced Tiles có thể khiến nhiều lần restart dừng ở các plateau tương tự nhau.

---

## 7. Local Beam Search với `k = 2`
### a. Ý tưởng
**Local Beam Search** không nên bắt đầu bằng hai initial khác nhau. Trong bài toán này, tôi tôn trọng initial state giống các thuật toán trước: tại bước đầu tiên chỉ có **một** trạng thái ban đầu.

Quy trình đúng với `k = 2`:
1. Bước 0: beam chỉ chứa initial state.
2. Từ initial, sinh toàn bộ successor hợp lệ.
3. Chọn tối đa `k = 2` successor theo phiên bản đang xét để tạo beam đầu tiên.
4. Từ thời điểm đã có beam, mỗi tầng tiếp theo sinh successor từ các state trong beam hiện tại rồi chọn lại tối đa `k` state.

Vì vậy **Beam 1** và **Beam 2** chỉ xuất hiện sau bước chọn beam đầu tiên. Chúng không phải là hai initial khác nhau, cũng không phải bước 1 và bước 2 của cùng một đường đi. Chúng là hai trạng thái song song được giữ lại ở cùng một tầng tìm kiếm.

### b. Hai phiên bản
1. **Local Beam - Chọn k tốt hơn**: chỉ xét successor tốt hơn trạng thái cha của nó, tức `h(child) < h(parent)`, rồi lấy tối đa `k` state tốt nhất trong tập này. Với case study hiện tại, ở bước đầu tiên initial chỉ có 1 successor thật sự tốt hơn, nên beam đầu tiên có kích thước `1/2`; đây là hệ quả của trạng thái đầu, không phải lỗi thuật toán.
2. **Local Beam - Chọn k tốt nhất**: xét toàn bộ successor của các state hiện tại, rồi chọn đúng `k` successor có `h(n)` thấp nhất nếu có đủ successor, không bắt buộc phải tốt hơn cha.

### c. Bốn đặc trưng
- **Tính đầy đủ**: Không. Beam có thể hội tụ vào vùng xấu và bỏ mất nhánh dẫn tới goal.
- **Tính tối ưu**: Không. Thuật toán không đảm bảo tìm đường đi ngắn nhất.
- **Độ phức tạp thời gian**: Xấp xỉ `O(t * k * b * cost(h))`. Với bài này `k = 2`.
- **Độ phức tạp không gian**: `O(k * b)` nếu lưu toàn bộ successor của beam ở mỗi vòng.

### d. Nhận xét hiệu năng
Local Beam Search thường mạnh hơn Hill Climbing đơn lẻ vì nó không phụ thuộc vào một trạng thái hiện tại duy nhất. Tuy nhiên, nếu hai beam trở nên giống nhau hoặc cùng bị hút vào một vùng heuristic xấu, thuật toán vẫn có thể thất bại. Manhattan Distance thường giúp việc chọn top-k có ý nghĩa hơn so với Misplaced Tiles vì thứ hạng successor được phân biệt tốt hơn.

---

## 8. Hướng dẫn chạy và quan sát giao diện
1. Chạy cell code để mở giao diện Tkinter.
2. Chọn thuật toán ở combobox `Algorithm`.
3. Chọn `Manhattan Distance` hoặc `Misplaced Tiles` ở combobox `Heuristic`.
4. Nhấn `Reset`, sau đó dùng `Next Step` hoặc `Auto Run`.
5. Ở Local Beam Search, bước đầu tiên chỉ hiển thị **Initial State**. Sau khi sinh successor và chọn beam, vùng **Current State** mới hiển thị Beam 1, Beam 2 nếu đã chọn đủ `k = 2`; vùng **Frontier / Neighbors** cho biết successor nào được sinh từ initial hoặc từ từng beam.


In [ ]:
# =========================================================================
# CHƯƠNG TRÌNH MÔ PHỎNG BUỔI 10: LOCAL SEARCH VARIANTS TRÊN 8-PUZZLE
# - Giao diện giữ cùng phong cách Buổi 9: Current State, Frontier/Neighbors,
#   Reset, Next Step, Auto Run và Speed Slider.
# - Stochastic Hill Climbing: chọn ngẫu nhiên trong tập neighbor tốt hơn.
# - Sideways Hill Climbing: chọn ngẫu nhiên trong tập neighbor tốt hơn hoặc bằng.
# - Random-Restart Hill Climbing: restart bằng các start 8-puzzle solvable.
# - Local Beam Search k=2: 2 phiên bản chọn k tốt hơn và chọn k tốt nhất.
# =========================================================================

import random
import tkinter as tk
from tkinter import ttk

PUZZLE_START = ((2, 8, 3), (1, 6, 4), (7, 0, 5))
PUZZLE_GOAL = ((1, 2, 3), (8, 0, 4), (7, 6, 5))
MOVES = [
    (-1, 0, 'Up'),
    (1, 0, 'Down'),
    (0, -1, 'Left'),
    (0, 1, 'Right'),
]
K_BEAM = 2
DEFAULT_MAX_STEPS = 150
DEFAULT_MAX_RESTART = 25
DEFAULT_SIDEWAYS_LIMIT = 20


# ── 8-Puzzle helpers ─────────────────────────────────────
def find_blank(state):
    for r in range(3):
        for c in range(3):
            if state[r][c] == 0:
                return r, c
    raise ValueError('State không có ô trống 0')


def get_neighbors(state):
    x, y = find_blank(state)
    result = []
    for dx, dy, action in MOVES:
        nx, ny = x + dx, y + dy
        if 0 <= nx < 3 and 0 <= ny < 3:
            new_state = [list(row) for row in state]
            new_state[x][y], new_state[nx][ny] = new_state[nx][ny], new_state[x][y]
            result.append((tuple(tuple(row) for row in new_state), action))
    return result


def manhattan_distance(state, goal=PUZZLE_GOAL):
    goal_pos = {}
    for r in range(3):
        for c in range(3):
            goal_pos[goal[r][c]] = (r, c)

    distance = 0
    for r in range(3):
        for c in range(3):
            value = state[r][c]
            if value != 0:
                gr, gc = goal_pos[value]
                distance += abs(r - gr) + abs(c - gc)
    return distance


def misplaced_tiles(state, goal=PUZZLE_GOAL):
    wrong = 0
    for r in range(3):
        for c in range(3):
            if state[r][c] != 0 and state[r][c] != goal[r][c]:
                wrong += 1
    return wrong


def random_puzzle_state(scramble_steps=25):
    state = PUZZLE_GOAL
    previous = None
    for _ in range(scramble_steps):
        choices = get_neighbors(state)
        if previous is not None and len(choices) > 1:
            choices = [item for item in choices if item[0] != previous]
        previous = state
        state, _ = random.choice(choices)
    return state


def node(state, heuristic, action='Start', status=''):
    return {
        'state': state,
        'h': heuristic(state),
        'action': action,
        'status': status,
    }


def is_goal(state):
    return state == PUZZLE_GOAL


def choose_heuristic(name):
    return misplaced_tiles if name == 'Misplaced Tiles' else manhattan_distance


# ── Local Search generators ──────────────────────────────
def hill_climbing_gen(start, heuristic, variant, max_steps=DEFAULT_MAX_STEPS, sideways_limit=DEFAULT_SIDEWAYS_LIMIT):
    current = start
    expanded = 0
    generated = 0
    sideways_used = 0
    visited = []

    for step in range(max_steps + 1):
        current_h = heuristic(current)
        visited.append(current)
        if is_goal(current):
            yield node(current, heuristic), [], visited, expanded, generated, [current], f'{variant} - GOAL'
            return

        raw_neighbors = get_neighbors(current)
        generated += len(raw_neighbors)
        candidates = []
        for state, action in raw_neighbors:
            h_value = heuristic(state)
            item = node(state, heuristic, action=action, status='Rejected')
            item['parent_h'] = current_h
            if variant == 'Stochastic HC':
                if h_value < current_h:
                    candidates.append(item)
            elif variant == 'Sideways HC':
                if h_value < current_h or (h_value == current_h and sideways_used < sideways_limit):
                    candidates.append(item)
            else:
                if h_value < current_h:
                    candidates.append(item)

        chosen = None
        if variant == 'Stochastic HC':
            if candidates:
                chosen = random.choice(candidates)
        elif variant == 'Sideways HC':
            improving = [item for item in candidates if item['h'] < current_h]
            pool = improving if improving else candidates
            if pool:
                chosen = random.choice(pool)
        else:
            if candidates:
                best_h = min(item['h'] for item in candidates)
                best = [item for item in candidates if item['h'] == best_h]
                chosen = random.choice(best)

        frontier = []
        for item in [node(state, heuristic, action=action, status='Rejected') for state, action in raw_neighbors]:
            if chosen and item['state'] == chosen['state']:
                item['status'] = 'Chosen'
                if variant == 'Sideways HC' and item['h'] == current_h:
                    item['status'] = 'Sideways'
            elif item['h'] < current_h:
                item['status'] = 'Better'
            elif item['h'] == current_h:
                item['status'] = 'Equal'
            frontier.append(item)

        if chosen is None:
            yield node(current, heuristic), frontier, visited, expanded, generated, None, f'{variant} FAILED - LOCAL OPTIMUM / PLATEAU'
            return

        expanded += 1
        if variant == 'Sideways HC' and chosen['h'] == current_h:
            sideways_used += 1
        else:
            sideways_used = 0

        mode = f'{variant} | sideways={sideways_used}/{sideways_limit}' if variant == 'Sideways HC' else variant
        yield node(current, heuristic), frontier, visited, expanded, generated, None, mode
        current = chosen['state']

    yield node(current, heuristic), [], visited, expanded, generated, None, f'{variant} FAILED - MAX STEPS'


def random_restart_hc_gen(heuristic, max_restart=DEFAULT_MAX_RESTART, max_steps=DEFAULT_MAX_STEPS):
    expanded = 0
    generated = 0
    explored = []
    best_state = PUZZLE_START
    best_h = heuristic(PUZZLE_START)

    for restart in range(max_restart + 1):
        current = PUZZLE_START if restart == 0 else random_puzzle_state(18 + restart % 12)
        start_label = 'Initial State' if restart == 0 else f'Random Start {restart}'

        for step in range(max_steps):
            current_h = heuristic(current)
            explored.append(current)
            if current_h < best_h:
                best_state = current
                best_h = current_h

            if is_goal(current):
                yield node(current, heuristic, action=start_label), [], explored, expanded, generated, [current], f'Random-Restart HC - GOAL at {start_label}'
                return

            raw_neighbors = get_neighbors(current)
            generated += len(raw_neighbors)
            frontier = [node(state, heuristic, action=action, status='Rejected') for state, action in raw_neighbors]
            better = [item for item in frontier if item['h'] < current_h]
            if not better:
                for item in frontier:
                    if item['h'] == current_h:
                        item['status'] = 'Equal'
                mode = f'Random-Restart HC | {start_label} stuck, best h={best_h}'
                yield node(current, heuristic, action=start_label), frontier, explored, expanded, generated, None, mode
                break

            best_h_next = min(item['h'] for item in better)
            best = [item for item in better if item['h'] == best_h_next]
            chosen = random.choice(best)
            for item in frontier:
                if item['state'] == chosen['state']:
                    item['status'] = 'Chosen'
                elif item['h'] < current_h:
                    item['status'] = 'Better'

            expanded += 1
            mode = f'Random-Restart HC | {start_label} | local step {step}'
            yield node(current, heuristic, action=start_label), frontier, explored, expanded, generated, None, mode
            current = chosen['state']

    yield node(best_state, heuristic, action='Best state found'), [], explored, expanded, generated, None, f'Random-Restart HC FAILED - max_restart={max_restart}, best h={best_h}' 

def beam_cards(states, heuristic, prefix='Current Beam'):
    cards = []
    for index, state in enumerate(states):
        label = 'Initial State' if len(states) == 1 and prefix == 'Current Beam' else f'{prefix} {index + 1}'
        item = node(state, heuristic, action=label, status='Chosen')
        item['beam_index'] = index
        item['role'] = 'beam'
        cards.append(item)
    return cards

def is_better_beam_mode(mode):
    # Tách logic khỏi so sánh nguyên chuỗi để tránh lỗi encoding/caption UI.
    return 'tốt hơn' in mode or 'tot hon' in mode.lower()


def local_beam_gen(initial_state, heuristic, mode, max_steps=DEFAULT_MAX_STEPS):
    states = [initial_state]
    expanded = 0
    generated = 0
    explored = []
    seen_beams = {tuple(states)}

    for step in range(max_steps + 1):
        explored.extend(states)
        if any(is_goal(state) for state in states):
            current_cards = beam_cards(states, heuristic)
            yield current_cards, [], explored, expanded, generated, states, f'{mode} - GOAL in beam'
            return

        better_mode = is_better_beam_mode(mode)
        raw_candidates = []
        candidate_id = 0
        for parent_index, parent in enumerate(states):
            parent_h = heuristic(parent)
            parent_label = 'Initial' if len(states) == 1 else f'Beam {parent_index + 1}'
            for child, action in get_neighbors(parent):
                child_h = heuristic(child)
                if better_mode and child_h >= parent_h:
                    continue
                item = node(child, heuristic, action=action, status='Rejected')
                item['candidate_id'] = candidate_id
                item['parent_index'] = parent_index
                item['parent_label'] = parent_label
                item['parent_h'] = parent_h
                item['source'] = parent_label
                raw_candidates.append(item)
                candidate_id += 1

        generated += len(raw_candidates)
        if not raw_candidates:
            reason = 'no better successors' if better_mode else 'no successors'
            yield beam_cards(states, heuristic), [], explored, expanded, generated, None, f'{mode} FAILED - {reason}'
            return

        raw_candidates.sort(key=lambda item: (item['h'], item['parent_index'], item['action'], item['state']))
        chosen = []
        seen_states = set()
        for item in raw_candidates:
            if item['state'] not in seen_states:
                chosen.append(item)
                seen_states.add(item['state'])
            if len(chosen) == K_BEAM:
                break

        # Chỉ đánh dấu đúng các candidate được chọn, không đánh dấu mọi bản sao có cùng state.
        chosen_ids = {item['candidate_id'] for item in chosen}
        frontier = []
        for item in raw_candidates:
            copy = dict(item)
            if copy['candidate_id'] in chosen_ids:
                copy['status'] = 'Chosen'
            elif copy['h'] < copy.get('parent_h', 999):
                copy['status'] = 'Better'
            frontier.append(copy)

        next_states = [item['state'] for item in chosen]
        next_signature = tuple(next_states)
        if next_signature == tuple(states):
            yield beam_cards(states, heuristic), frontier, explored, expanded, generated, None, f'{mode} FAILED - beam unchanged'
            return
        if next_signature in seen_beams:
            yield beam_cards(states, heuristic), frontier, explored, expanded, generated, None, f'{mode} FAILED - beam repeated'
            return

        expanded += len(states)
        beam_size_note = f'beam size={len(next_states)}/{K_BEAM}'
        yield beam_cards(states, heuristic), frontier, explored, expanded, generated, None, f'{mode} | {beam_size_note}'
        seen_beams.add(next_signature)
        states = next_states

    yield beam_cards(states, heuristic), [], explored, expanded, generated, None, f'{mode} FAILED - MAX STEPS' 

# ── GUI widgets ──────────────────────────────────────────
class ScrollableFrame(tk.Frame):
    def __init__(self, parent, height=230):
        super().__init__(parent)
        self.canvas = tk.Canvas(self, height=height, highlightthickness=0)
        self.scroll_x = tk.Scrollbar(self, orient='horizontal', command=self.canvas.xview)
        self.inner = tk.Frame(self.canvas)
        self.inner.bind('<Configure>', lambda event: self.canvas.configure(scrollregion=self.canvas.bbox('all')))
        self.canvas.create_window((0, 0), window=self.inner, anchor='nw')
        self.canvas.configure(xscrollcommand=self.scroll_x.set)
        self.canvas.pack(side='top', fill='both', expand=True)
        self.scroll_x.pack(side='bottom', fill='x')


class PuzzleGUI:
    def __init__(self, root):
        self.root = root
        self.gen = None
        self.running = False
        self.step_count = 0

        root.title('AI Search - Buổi 10 Local Search Variants on 8-Puzzle')
        sw, sh = root.winfo_screenwidth(), root.winfo_screenheight()
        w, h = min(1400, int(sw * .85)), min(900, int(sh * .85))
        root.geometry(f'{w}x{h}+{(sw-w)//2}+{(sh-h)//2}')
        root.minsize(940, 620)

        self.info = tk.Label(root, text='8-Puzzle Local Search - click Reset, then Next Step or Auto Run', font=('Arial', 14, 'bold'))
        self.info.pack(pady=6)

        self.cur_frame = tk.LabelFrame(root, text='Current State (Popped Node / Current Beam)', padx=8, pady=8)
        self.cur_frame.pack(side='top', fill='x', padx=10, pady=4)

        self.fl_frame = tk.LabelFrame(root, text='Frontier / Neighbors', padx=8, pady=8)
        self.fl_frame.pack(side='top', fill='both', expand=True, padx=10, pady=4)
        self.frontier_sf = ScrollableFrame(self.fl_frame, 260)
        self.frontier_sf.pack(fill='both', expand=True)

        bottom = tk.Frame(root)
        bottom.pack(side='bottom', fill='x', pady=8)
        center = tk.Frame(bottom)
        center.pack(anchor='center')

        tk.Label(center, text='Problem:', font=('Arial', 10, 'bold')).pack(side='left', padx=(5, 2))
        self.prob_cb = ttk.Combobox(center, values=['8-Puzzle'], state='readonly', width=12)
        self.prob_cb.set('8-Puzzle')
        self.prob_cb.pack(side='left', padx=5)

        tk.Label(center, text='Algorithm:', font=('Arial', 10, 'bold')).pack(side='left', padx=(12, 2))
        self.algo_cb = ttk.Combobox(center, values=[
            'Steepest-Ascent HC',
            'Stochastic HC',
            'Sideways HC',
            'Random-Restart HC',
            'Local Beam - Chọn k tốt hơn',
            'Local Beam - Chọn k tốt nhất',
        ], state='readonly', width=26)
        self.algo_cb.set('Stochastic HC')
        self.algo_cb.pack(side='left', padx=5)

        tk.Label(center, text='Heuristic:', font=('Arial', 10, 'bold')).pack(side='left', padx=(12, 2))
        self.heuristic_cb = ttk.Combobox(center, values=['Manhattan Distance', 'Misplaced Tiles'], state='readonly', width=18)
        self.heuristic_cb.set('Manhattan Distance')
        self.heuristic_cb.pack(side='left', padx=5)

        tk.Label(center, text='max_restart').pack(side='left', padx=(10, 2))
        self.max_restart_var = tk.IntVar(value=DEFAULT_MAX_RESTART)
        tk.Spinbox(center, from_=0, to=300, increment=5, textvariable=self.max_restart_var, width=5).pack(side='left')

        tk.Label(center, text='sideways').pack(side='left', padx=(10, 2))
        self.sideways_var = tk.IntVar(value=DEFAULT_SIDEWAYS_LIMIT)
        tk.Spinbox(center, from_=0, to=200, increment=5, textvariable=self.sideways_var, width=5).pack(side='left')

        tk.Button(center, text='Reset', width=8, command=self.reset_search, bg='#FF5722', fg='white').pack(side='left', padx=(10, 5))
        tk.Button(center, text='Next Step', width=12, command=self.next_step).pack(side='left', padx=5)
        self.auto_btn = tk.Button(center, text='Auto Run', width=12, command=self.toggle_auto)
        self.auto_btn.pack(side='left', padx=5)

        self.speed_var = tk.IntVar(value=300)
        tk.Label(center, text='Speed:').pack(side='left', padx=(12, 2))
        tk.Scale(center, from_=10, to=1000, orient='horizontal', variable=self.speed_var, showvalue=False, length=120).pack(side='left')

        self.reset_search()

    def current_heuristic(self):
        return choose_heuristic(self.heuristic_cb.get())

    def reset_search(self):
        self.running = False
        self.step_count = 0
        self.auto_btn.config(text='Auto Run')
        self.clear(self.cur_frame)
        self.clear(self.frontier_sf.inner)

        heuristic = self.current_heuristic()
        algo = self.algo_cb.get()
        if algo == 'Random-Restart HC':
            self.gen = random_restart_hc_gen(heuristic, self.max_restart_var.get(), DEFAULT_MAX_STEPS)
            self.fl_frame.config(text='Frontier / Neighbors - restart attempts')
            self.draw_state(self.cur_frame, PUZZLE_START, 'Reference Start State').pack(side='left', padx=8)
        elif algo.startswith('Local Beam'):
            self.gen = local_beam_gen(PUZZLE_START, heuristic, algo, DEFAULT_MAX_STEPS)
            self.fl_frame.config(text='All Beam Successors (Chosen: Green, Better: Blue, Rejected: Red)')
            for card in beam_cards([PUZZLE_START], heuristic):
                self.draw_state(self.cur_frame, card['state'], f"{card['action']}\nh(n) = {card['h']}").pack(side='left', padx=8)
        else:
            self.gen = hill_climbing_gen(PUZZLE_START, heuristic, algo, DEFAULT_MAX_STEPS, self.sideways_var.get())
            self.fl_frame.config(text='Immediate Neighbors Evaluated (Chosen: Green, Better: Blue, Equal: Yellow, Rejected: Red)')
            self.draw_state(self.cur_frame, PUZZLE_START, 'Initial Start State').pack()

        self.info.config(text='Ready! Click Next Step to Start')

    def draw_state(self, parent, state, label_text=None, bg_color='white'):
        container = tk.Frame(parent, bg=bg_color, bd=2, relief='groove')
        grid_frame = tk.Frame(container, bg=bg_color)
        grid_frame.pack(padx=4, pady=4)

        for r in range(3):
            for c in range(3):
                value = state[r][c]
                cell_bg = '#4CAF50' if value == 0 else 'white'
                tk.Label(
                    grid_frame,
                    text=str(value) if value else '',
                    width=4,
                    height=2,
                    font=('Arial', 13, 'bold'),
                    relief='solid',
                    bg=cell_bg,
                ).grid(row=r, column=c)

        if label_text:
            tk.Label(container, text=label_text, font=('Arial', 10, 'bold'), fg='black', bg=bg_color, justify='center').pack(pady=4)
        return container

    def clear(self, frame):
        for widget in frame.winfo_children():
            widget.destroy()

    def render(self, current, frontier, explored, expanded, generated, path, mode):
        self.clear(self.cur_frame)
        self.clear(self.frontier_sf.inner)

        current_items = current if isinstance(current, list) else [current]
        for item in current_items:
            label = f"{item.get('action', 'Current')}\nh(n) = {item['h']}"
            if item.get('role') == 'beam':
                label += "\n(state kept in k=2 beam)"
            self.draw_state(self.cur_frame, item['state'], label).pack(side='left', padx=8)

        for index, item in enumerate(frontier):
            if 'parent_index' in item:
                source_label = item.get('parent_label', f"Beam {item['parent_index'] + 1}")
                info = (
                    f"From {source_label}: {item.get('action', '')}\n"
                    f"h(parent)={item.get('parent_h')} -> h(child)={item['h']}"
                )
            elif item.get('status') == 'Kept':
                info = f"{item.get('action', '')}\nh(n) = {item['h']}\n[kept to preserve k=2]"
            else:
                info = f"{item.get('action', '')}\nh(n) = {item['h']}"
            status = item.get('status', '')
            bg_color = 'white'
            if status == 'Chosen':
                bg_color = '#C8E6C9'
                info += '\n[CHOSEN]'
            elif status == 'Sideways':
                bg_color = '#FFF59D'
                info += '\n[SIDEWAYS]'
            elif status == 'Better':
                bg_color = '#BBDEFB'
                info += '\n[Better]'
            elif status == 'Equal':
                bg_color = '#FFF9C4'
                info += '\n[Equal]'
            elif status == 'Kept':
                bg_color = '#E0E0E0'
                info += '\n[KEPT]'
            elif status == 'Rejected':
                bg_color = '#FFCDD2'
                info += '\n[Rejected]'

            self.draw_state(self.frontier_sf.inner, item['state'], info, bg_color).grid(row=0, column=index, padx=8, pady=8)

        status = f'[{mode}] Step: {self.step_count} | Expanded: {expanded} | Generated: {generated} | Explored: {len(explored)}'
        if path:
            status += ' | 🎉 REACHED GOAL'
        if 'FAILED' in mode:
            status += ' | ❌ THUẬT TOÁN BỊ KẸT!'
        self.info.config(text=status)

    def next_step(self):
        try:
            current, frontier, explored, expanded, generated, path, mode = next(self.gen)
            self.step_count += 1
            self.render(current, frontier, explored, expanded, generated, path, mode)
            if path or 'FAILED' in mode:
                self.running = False
                self.auto_btn.config(text='Auto Run')
        except StopIteration:
            self.running = False
            self.auto_btn.config(text='Done')

    def toggle_auto(self):
        self.running = not self.running
        self.auto_btn.config(text='Pause' if self.running else 'Auto Run')
        if self.running:
            self.auto_tick()

    def auto_tick(self):
        if not self.running:
            return
        self.next_step()
        if self.running:
            self.root.after(self.speed_var.get(), self.auto_tick)


root = tk.Tk()
app = PuzzleGUI(root)
root.mainloop()
